In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "proyecto_olist")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_orders = spark.table(f"{catalogo}.{esquema_source}.orders")

In [0]:
df_orders_clean = (
    df_orders
    .select(
        col("order_id").cast("string").alias("order_id"),
        col("customer_id").cast("string").alias("customer_id"),
        col("order_status").cast("string").alias("order_status"),
        to_timestamp("order_purchase_timestamp").alias("order_purchase_timestamp"),
        to_timestamp("order_approved_at").alias("order_approved_at"),
        to_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
        to_timestamp("order_delivered_customer_date").alias("order_delivered_customer_date"),
        to_timestamp("order_estimated_delivery_date").alias("order_estimated_delivery_date")
    )
    .filter(col("order_id").isNotNull())
    .withColumn("silver_insert_date", current_timestamp())
)

In [0]:
df_orders_clean.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.orders")

In [0]:
print("Tabla silver.orders procesada.")